In [20]:
import os
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Dict
import numpy as np

def find_config_file(folder_path: str) -> Optional[str]:
    """
    Finds a configuration file (ending with .txt) within the 'configs' subfolder.
    """
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a 'key = value', 'key value', or 'key: value' configuration file into a dictionary.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                separator = None
                if ':' in line:
                    separator = ':'
                elif '=' in line:
                    separator = '='

                if separator:
                    parts = line.split(separator, 1)
                else:
                    parts = line.split(None, 1)

                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params

def extract_run_number(folder_name: str) -> int:
    """Extracts the run number from folder name like 'run_01_timestamp' or 'run_timestamp'."""
    import re
    match = re.search(r'run_(\d+)_', folder_name)
    if match:
        return int(match.group(1))
    return 1  # Default to 1 if not found

def find_timing_file(run_folder_path: str, sim_type: str) -> Optional[str]:
    """Finds the ...trace_matched_timing.csv file for a given run."""
    sim_output_dir = os.path.join(run_folder_path, sim_type.lower())
    if not os.path.isdir(sim_output_dir):
        return None
    for f in os.listdir(sim_output_dir):
        if 'trace_matched_timing.csv' in f:
            return os.path.join(sim_output_dir, f)
    return None

# --- Collective Info Parsing ---
def parse_collectives_log(log_path: str) -> Dict[str, Dict]:
    """Parses a duplicate_collectives.log file to extract signatures."""
    collective_info = {}
    try:
        with open(log_path, 'r') as f:
            lines = f.readlines()
            i = 0
            while i < len(lines):
                line = lines[i]
                workload_match = re.match(r'^Workload: (\S+)', line)
                if workload_match:
                    current_workload = workload_match.group(1)
                    # Look for signature on the next line
                    if (i + 1 < len(lines)) and (signature_match := re.match(r'^\s+Signature: \((.*)\)', lines[i+1])):
                        sig_content = signature_match.group(1).strip()
                        # Split signature into its three parts
                        parts = sig_content.rsplit(', ', 2)
                        if len(parts) == 3:
                            npu_tuples, comm_type, comm_size = parts
                            collective_info[current_workload] = {
                                'npu_tuples': npu_tuples.strip(),
                                'comm_type': comm_type.strip(),
                                'comm_size': comm_size.strip()
                            }
                i += 1
    except FileNotFoundError:
        print(f"Warning: Collectives log file not found at {log_path}")
    except Exception as e:
        print(f"Error parsing collectives log {log_path}: {e}")
    return collective_info


In [21]:
import re
import pandas as pd
from plotly.subplots import make_subplots

# --- Configuration ---
base_comparison_folders = [
    '/app/astra-sim/upc/output/comparison_run/Dragonfly/multiple_collectives_tp',
]
comparison_plot_metric = 'avg'  # Can be 'avg' or 'max'

# --- Helper Functions ---
cc_modes = {0: "PFC", 1: "DCQCN", 3: "HPCC", 7: "TIMELY", 8: "DCTCP", 10: "HPCC-PINT"}
workload_info_regex = re.compile(r'([a-zA-Z_]+)_size_(\d+)_(\d+)')

def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into seconds."""
    if not time_str: return 0.0
    try:
        h, m, s = map(float, time_str.split(':'))
        return h * 3600 + m * 60 + s
    except (ValueError, IndexError):
        return 0.0

def process_simulation(folder: str, sim_type: str, summary_params: dict, topo_file: str, run_name: str, workload_name: str, comm_size: int, group_id: int, run_number: int) -> Optional[dict]:
    """Processes a single simulation run, extracts timing data, and returns a result dictionary."""
    timing_file = find_timing_file(folder, sim_type)
    if not timing_file:
        return None

    try:
        df = pd.read_csv(timing_file)
        if sim_type == 'ns3' and 'node_name' in df.columns:
            df = df[df['node_name'] != 'dummy_node'].copy()

        time_col = 'callback_tick'
        sys_id_col = 'sys_id'
        
        if time_col not in df.columns or sys_id_col not in df.columns:
            return None

        # Filter out invalid times
        df = df[df[time_col] > 100].copy()
        
        if df.empty:
            return None
        
        # Get the LAST (maximum) time for each NPU - this is when that NPU completes its last communication
        npu_completion_times = df.groupby(sys_id_col)[time_col].max()
        
        if npu_completion_times.empty:
            return None

        workload_name = summary_params.get('collective', 'N/A').strip()
        
        # Extract ECMP seed for NS3
        ecmp_seed = None
        if sim_type == 'ns3':
            ecmp_seed_str = summary_params.get('ns3 ecmp seed override', None)
            if ecmp_seed_str and ecmp_seed_str != 'None':
                try:
                    ecmp_seed = int(ecmp_seed_str)
                except ValueError:
                    pass

        return {
            'workload': workload_name,
            'comm_size': comm_size,
            'comm_type': workload_name,
            'group_id': group_id,
            'run_number': run_number,
            'ecmp_seed': ecmp_seed,
            'npu_count': summary_params.get('npus count', 'N/A'),
            'topology': ('_'.join(topo_file.split('_')[1:]).rsplit('.', 1)[0]),
            'sim_type': sim_type.upper(),
            'run_name': run_name,
            'avg_time': npu_completion_times.mean(),  # Average completion time across NPUs
            'max_time': npu_completion_times.max(),   # Max completion time (overall completion)
            'min_time': npu_completion_times.min(),   # Min completion time (fastest NPU)
            'std_dev': npu_completion_times.std(),    # Std dev shows load imbalance
            'execution_time': parse_runtime(summary_params.get('total runtime', '0:0:0.0')),
            'path': folder
        }
    except Exception as e:
        print(f"Error processing {sim_type} in {folder}: {e}")
        return None

# --- Data Collection Logic ---
all_run_folders = []
for base_folder in base_comparison_folders:
    for workload_folder in os.listdir(base_folder):
        workload_path = os.path.join(base_folder, workload_folder)
        if os.path.isdir(workload_path):
            all_run_folders.extend([os.path.join(workload_path, d) for d in os.listdir(workload_path) if os.path.isdir(os.path.join(workload_path, d))])

comparison_results = []
for folder in sorted(all_run_folders):
    run_summary_path = os.path.join(folder, 'run_summary.txt')
    if not os.path.exists(run_summary_path):
        continue

    summary_params = parse_config(run_summary_path)
    
    # Extract run number from folder name or summary
    folder_name = os.path.basename(folder)
    run_number = extract_run_number(folder_name)
    if 'run number' in summary_params:
        try:
            run_number = int(summary_params['run number'])
        except ValueError:
            pass
    
    # Extract info from workload folder name
    workload_folder_name = os.path.basename(os.path.dirname(folder))
    workload_name, comm_size, group_id = 'N/A', -1, -1
    workload_match = workload_info_regex.match(workload_folder_name)
    if workload_match:
        workload_name = workload_match.group(1)
        comm_size = int(workload_match.group(2))
        group_id = int(workload_match.group(3))

    # Process all potential simulation types in the folder
    sim_dirs = [d for d in os.listdir(folder) if os.path.isdir(os.path.join(folder, d)) and d in ['analytical_unaware', 'g2', 'ns3']]

    for sim_type in sim_dirs:
        topo_file = "N/A"
        run_name = f"{sim_type.replace('_', ' ').title()}"

        if sim_type != 'analytical_unaware':
            topo_key = 'g2 topology file override' if sim_type == 'g2' else 'ns3 topology file override'
            topo_path = summary_params.get(topo_key, '')
            if 'all_paths' in topo_path: continue
            if not topo_path: continue
            topo_file = os.path.basename(topo_path)

            if sim_type == 'ns3':
                ns3_config_file = find_config_file(folder)
                if ns3_config_file:
                    ns3_params = parse_config(ns3_config_file)
                    run_name = (
                        f"NS3 (cc:{cc_modes.get(int(ns3_params.get('cc_mode', -1)), 'N/A')}, "
                        f"win:{ns3_params.get('has_win', 'N/A')}, adapt:{ns3_params.get('var_win', 'N/A')}, "
                        f"buf:{ns3_params.get('buffer_size', 'N/A')}, size:{ns3_params.get('packet_payload_size', 'N/A')})"
                    )
            else: # G2
                run_name = "G2"

        result = process_simulation(folder, sim_type, summary_params, topo_file, run_name, workload_name, comm_size, group_id, run_number)
        if result:
            comparison_results.append(result)

print(f"Collected {len(comparison_results)} simulation results")
print(f"Note: Times represent the last (completion) time for each NPU")


Collected 154 simulation results
Note: Times represent the last (completion) time for each NPU


In [22]:
# --- Analysis and Plotting ---
if comparison_results:
    comp_df = pd.DataFrame(comparison_results)

    # Map for collective communication types
    comm_type_map = {
        "all_reduce": "ALL_REDUCE", "reduce": "REDUCE", "all_gather": "ALL_GATHER", "gather": "GATHER",
        "scatter": "SCATTER", "broadcast": "BROADCAST", "all_to_all": "ALL_TO_ALL", "reduce_scatter": "REDUCE_SCATTER",
        "reduce_scatter_block": "REDUCE_SCATTER_BLOCK", "barrier": "BARRIER"
    }
    comp_df['comm_type_str'] = comp_df['comm_type'].astype(str).map(comm_type_map).fillna(comp_df['comm_type'])

    print(f"\n{'='*80}")
    print(f"RANDOMNESS ANALYSIS: Effect of Multiple Runs on G2 vs NS3")
    print(f"{'='*80}\n")
    
    # --- Summary Statistics ---
    print(f"Total runs collected: {len(comp_df)}")
    print(f"Unique workloads: {comp_df['workload'].nunique()}")
    print(f"Simulation types: {comp_df['sim_type'].unique()}")
    print(f"Run numbers range: {comp_df['run_number'].min()} to {comp_df['run_number'].max()}")
    
    # Analyze both avg and max times
    metrics_to_analyze = [
        ('avg_time', 'Average NPU Completion Time'),
        ('max_time', 'Overall Completion Time (Slowest NPU)')
    ]
    
    for metric_col, metric_name in metrics_to_analyze:
        print(f"\n{'='*80}")
        print(f"ANALYSIS FOR: {metric_name}")
        print(f"{'='*80}")
        
        # Group by workload, group_id, topology, and sim_type to analyze multiple runs
        grouped_stats = []
        for group_keys, group_df in comp_df.groupby(['workload', 'group_id', 'topology', 'sim_type']):
            wl, gid, topo, sim = group_keys
            times = group_df[metric_col]
            
            grouped_stats.append({
                'Workload': wl,
                'Group ID': gid,
                'Topology': topo,
                'Sim Type': sim,
                'Num Runs': len(group_df),
                'Mean Time (ms)': times.mean() / 1_000_000,
                'Std Dev (ms)': times.std() / 1_000_000,
                'Min Time (ms)': times.min() / 1_000_000,
                'Max Time (ms)': times.max() / 1_000_000,
                'CV (%)': (times.std() / times.mean() * 100) if times.mean() > 0 else 0
            })
        
        stats_df = pd.DataFrame(grouped_stats)
        
        print(f"\n--- Coefficient of Variation (CV) Analysis for {metric_name} ---")
        print("CV shows relative variability: higher CV = more randomness\n")
        
        for sim_type in stats_df['Sim Type'].unique():
            sim_stats = stats_df[stats_df['Sim Type'] == sim_type]
            print(f"{sim_type}:")
            print(f"  Average CV: {sim_stats['CV (%)'].mean():.2f}%")
            print(f"  Max CV: {sim_stats['CV (%)'].max():.2f}%")
            print(f"  Min CV: {sim_stats['CV (%)'].min():.2f}%")
            print()

        # --- Create Randomness Comparison Plots ---
        unique_configs = comp_df.groupby(['workload', 'group_id', 'topology']).size().reset_index()[['workload', 'group_id', 'topology']]
        
        print(f"\n--- Generating Randomness Comparison Plots for {metric_name} ({len(unique_configs)} configurations) ---\n")
        
        for idx, config in unique_configs.head(10).iterrows():  # Plot first 10 configurations
            wl, gid, topo = config['workload'], config['group_id'], config['topology']
            
            config_df = comp_df[
                (comp_df['workload'] == wl) &
                (comp_df['group_id'] == gid) &
                (comp_df['topology'] == topo)
            ]
            
            g2_runs = config_df[config_df['sim_type'] == 'G2'].sort_values('run_number')
            ns3_runs = config_df[config_df['sim_type'] == 'NS3'].sort_values('run_number')
            
            if g2_runs.empty or ns3_runs.empty:
                continue
            
            # Create subplots
            fig = make_subplots(
                rows=2, cols=2,
                subplot_titles=(
                    f'Run-by-Run Comparison ({metric_name})',
                    f'Distribution Comparison',
                    'Coefficient of Variation',
                    'Per-NPU Completion Times'
                ),
                specs=[[{"secondary_y": False}, {"secondary_y": False}],
                       [{"secondary_y": False}, {"secondary_y": False}]]
            )
            
            # 1. Run-by-Run Line Plot
            fig.add_trace(go.Scatter(
                x=g2_runs['run_number'],
                y=g2_runs[metric_col] / 1_000_000,
                mode='lines+markers',
                name='G2',
                line=dict(color='blue', width=2),
                marker=dict(size=8)
            ), row=1, col=1)
            
            fig.add_trace(go.Scatter(
                x=ns3_runs['run_number'],
                y=ns3_runs[metric_col] / 1_000_000,
                mode='lines+markers',
                name='NS3',
                line=dict(color='red', width=2),
                marker=dict(size=8)
            ), row=1, col=1)
            
            # 2. Box Plot for Distribution
            fig.add_trace(go.Box(
                y=g2_runs[metric_col] / 1_000_000,
                name='G2',
                marker_color='blue',
                boxmean='sd'
            ), row=1, col=2)
            
            fig.add_trace(go.Box(
                y=ns3_runs[metric_col] / 1_000_000,
                name='NS3',
                marker_color='red',
                boxmean='sd'
            ), row=1, col=2)
            
            # 3. Coefficient of Variation Bar Chart
            g2_cv = (g2_runs[metric_col].std() / g2_runs[metric_col].mean() * 100) if g2_runs[metric_col].mean() > 0 else 0
            ns3_cv = (ns3_runs[metric_col].std() / ns3_runs[metric_col].mean() * 100) if ns3_runs[metric_col].mean() > 0 else 0
            
            fig.add_trace(go.Bar(
                x=['G2', 'NS3'],
                y=[g2_cv, ns3_cv],
                marker_color=['blue', 'red'],
                text=[f'{g2_cv:.2f}%', f'{ns3_cv:.2f}%'],
                textposition='auto',
                showlegend=False
            ), row=2, col=1)
            
            # 4. Per-NPU Completion Times (read timing files and get last time per NPU for all runs)
            all_g2_npu_completion = []
            all_ns3_npu_completion = []
            
            for _, g2_run in g2_runs.iterrows():
                timing_file = find_timing_file(g2_run['path'], 'G2')
                if timing_file:
                    try:
                        df_timing = pd.read_csv(timing_file)
                        df_timing = df_timing[df_timing['callback_tick'] > 100]
                        # Get last (max) time for each NPU
                        npu_completion = df_timing.groupby('sys_id')['callback_tick'].max().values / 1_000_000
                        all_g2_npu_completion.extend(npu_completion)
                    except:
                        pass
            
            for _, ns3_run in ns3_runs.iterrows():
                timing_file = find_timing_file(ns3_run['path'], 'NS3')
                if timing_file:
                    try:
                        df_timing = pd.read_csv(timing_file)
                        if 'node_name' in df_timing.columns:
                            df_timing = df_timing[df_timing['node_name'] != 'dummy_node']
                        df_timing = df_timing[df_timing['callback_tick'] > 100]
                        # Get last (max) time for each NPU
                        npu_completion = df_timing.groupby('sys_id')['callback_tick'].max().values / 1_000_000
                        all_ns3_npu_completion.extend(npu_completion)
                    except:
                        pass
            
            if all_g2_npu_completion:
                fig.add_trace(go.Box(
                    y=all_g2_npu_completion,
                    name='G2 NPUs',
                    marker_color='lightblue',
                    boxmean='sd'
                ), row=2, col=2)
            
            if all_ns3_npu_completion:
                fig.add_trace(go.Box(
                    y=all_ns3_npu_completion,
                    name='NS3 NPUs',
                    marker_color='lightcoral',
                    boxmean='sd'
                ), row=2, col=2)
            
            # Update axes labels
            fig.update_xaxes(title_text="Run Number", row=1, col=1)
            fig.update_yaxes(title_text="Time (ms)", row=1, col=1)
            fig.update_yaxes(title_text="Time (ms)", row=1, col=2)
            fig.update_yaxes(title_text="CV (%)", row=2, col=1)
            fig.update_yaxes(title_text="Completion Time (ms)", row=2, col=2)
            
            # Add statistics to title
            g2_mean = g2_runs[metric_col].mean() / 1_000_000
            ns3_mean = ns3_runs[metric_col].mean() / 1_000_000
            g2_std = g2_runs[metric_col].std() / 1_000_000
            ns3_std = ns3_runs[metric_col].std() / 1_000_000
            divergence = (ns3_mean - g2_mean) / g2_mean * 100
            
            title_text = (
                f"<b>{wl} (Group: {gid}, Topology: {topo})</b><br>"
                f"{metric_name}: G2: {g2_mean:.2f}±{g2_std:.2f} ms (CV: {g2_cv:.2f}%) | "
                f"NS3: {ns3_mean:.2f}±{ns3_std:.2f} ms (CV: {ns3_cv:.2f}%) | "
                f"Divergence: {divergence:+.1f}%"
            )
            
            fig.update_layout(
                title_text=title_text,
                height=800,
                margin=dict(t=150),
                showlegend=True
            )
            
            fig.show()
    
    print(f"\n{'='*80}")
    print("Analysis complete!")
    print(f"{'='*80}\n")

else:
    print("No comparison results to process.")



RANDOMNESS ANALYSIS: Effect of Multiple Runs on G2 vs NS3

Total runs collected: 154
Unique workloads: 3
Simulation types: ['ANALYTICAL_UNAWARE' 'G2' 'NS3']
Run numbers range: 1 to 20

ANALYSIS FOR: Average NPU Completion Time

--- Coefficient of Variation (CV) Analysis for Average NPU Completion Time ---
CV shows relative variability: higher CV = more randomness

ANALYTICAL_UNAWARE:
  Average CV: 0.00%
  Max CV: 0.00%
  Min CV: 0.00%

G2:
  Average CV: 13.61%
  Max CV: 14.79%
  Min CV: 12.53%

NS3:
  Average CV: 11.29%
  Max CV: 12.28%
  Min CV: 10.30%


--- Generating Randomness Comparison Plots for Average NPU Completion Time (6 configurations) ---




ANALYSIS FOR: Overall Completion Time (Slowest NPU)

--- Coefficient of Variation (CV) Analysis for Overall Completion Time (Slowest NPU) ---
CV shows relative variability: higher CV = more randomness

ANALYTICAL_UNAWARE:
  Average CV: 0.00%
  Max CV: 0.00%
  Min CV: 0.00%

G2:
  Average CV: 16.18%
  Max CV: 19.93%
  Min CV: 13.68%

NS3:
  Average CV: 14.01%
  Max CV: 17.47%
  Min CV: 10.54%


--- Generating Randomness Comparison Plots for Overall Completion Time (Slowest NPU) (6 configurations) ---




Analysis complete!



In [23]:
# --- Statistical Summary of Randomness Effects ---

print(f"\n{'='*80}")
print("DETAILED STATISTICAL ANALYSIS: Average vs Completion Time")
print(f"{'='*80}\n")

# Analyze both metrics
for metric_col, metric_name in [('avg_time', 'Average Time'), ('max_time', 'Completion Time')]:
    print(f"\n{'='*40}")
    print(f"METRIC: {metric_name}")
    print(f"{'='*40}\n")
    
    # Create comparison table
    comparison_data = []
    for group_keys, group_df in comp_df.groupby(['workload', 'group_id', 'topology']):
        wl, gid, topo = group_keys
        
        g2_runs = group_df[group_df['sim_type'] == 'G2']
        ns3_runs = group_df[group_df['sim_type'] == 'NS3']
        
        if g2_runs.empty or ns3_runs.empty:
            continue
        
        g2_times = g2_runs[metric_col]
        ns3_times = ns3_runs[metric_col]
        
        comparison_data.append({
            'Workload': wl,
            'Group': gid,
            'Topology': topo,
            'G2 Mean (ms)': g2_times.mean() / 1_000_000,
            'G2 Std (ms)': g2_times.std() / 1_000_000,
            'G2 CV (%)': (g2_times.std() / g2_times.mean() * 100) if g2_times.mean() > 0 else 0,
            'G2 Runs': len(g2_runs),
            'NS3 Mean (ms)': ns3_times.mean() / 1_000_000,
            'NS3 Std (ms)': ns3_times.std() / 1_000_000,
            'NS3 CV (%)': (ns3_times.std() / ns3_times.mean() * 100) if ns3_times.mean() > 0 else 0,
            'NS3 Runs': len(ns3_runs),
            'Mean Divergence (%)': ((ns3_times.mean() - g2_times.mean()) / g2_times.mean() * 100),
            'Best NS3 vs G2 Mean (%)': ((ns3_times.min() - g2_times.mean()) / g2_times.mean() * 100),
            'Worst NS3 vs G2 Mean (%)': ((ns3_times.max() - g2_times.mean()) / g2_times.mean() * 100),
        })

    comparison_summary_df = pd.DataFrame(comparison_data)

    # Display summary statistics
    print(f"--- Summary Statistics for {metric_name} ---\n")
    print(f"G2 Variability:")
    print(f"  Average CV: {comparison_summary_df['G2 CV (%)'].mean():.3f}%")
    print(f"  Std Dev of CV: {comparison_summary_df['G2 CV (%)'].std():.3f}%")
    print(f"  Max CV: {comparison_summary_df['G2 CV (%)'].max():.3f}%")
    print()
    print(f"NS3 Variability:")
    print(f"  Average CV: {comparison_summary_df['NS3 CV (%)'].mean():.3f}%")
    print(f"  Std Dev of CV: {comparison_summary_df['NS3 CV (%)'].std():.3f}%")
    print(f"  Max CV: {comparison_summary_df['NS3 CV (%)'].max():.3f}%")
    print()
    print(f"Divergence Analysis:")
    print(f"  Mean Divergence: {comparison_summary_df['Mean Divergence (%)'].mean():.2f}%")
    print(f"  Best Case (Best NS3 vs G2): {comparison_summary_df['Best NS3 vs G2 Mean (%)'].mean():.2f}%")
    print(f"  Worst Case (Worst NS3 vs G2): {comparison_summary_df['Worst NS3 vs G2 Mean (%)'].mean():.2f}%")
    print()

    # Display sorted table
    print(f"\n--- Top 10 Configurations by NS3 Variability (CV) for {metric_name} ---")
    display(comparison_summary_df.sort_values('NS3 CV (%)', ascending=False).head(10))

    # Create comparison visualization
    fig = go.Figure()

    fig.add_trace(go.Histogram(
        x=comparison_summary_df['G2 CV (%)'],
        name='G2',
        opacity=0.7,
        marker_color='blue',
        nbinsx=20
    ))

    fig.add_trace(go.Histogram(
        x=comparison_summary_df['NS3 CV (%)'],
        name='NS3',
        opacity=0.7,
        marker_color='red',
        nbinsx=20
    ))

    fig.update_layout(
        title=f'Distribution of Coefficient of Variation: G2 vs NS3 ({metric_name})<br><sub>Lower CV = More Consistent, Higher CV = More Random</sub>',
        xaxis_title='Coefficient of Variation (%)',
        yaxis_title='Frequency',
        barmode='overlay',
        height=500
    )

    fig.show()

# --- Comparison between Average and Max metrics ---
print(f"\n{'='*80}")
print("COMPARISON: Average Time vs Completion Time (Max)")
print(f"{'='*80}\n")

# Calculate differences between avg and max for each sim type
comparison_avg_max = []
for group_keys, group_df in comp_df.groupby(['workload', 'group_id', 'topology', 'sim_type']):
    wl, gid, topo, sim = group_keys
    
    avg_mean = group_df['avg_time'].mean() / 1_000_000
    max_mean = group_df['max_time'].mean() / 1_000_000
    diff_pct = ((max_mean - avg_mean) / avg_mean * 100) if avg_mean > 0 else 0
    
    comparison_avg_max.append({
        'Workload': wl,
        'Group': gid,
        'Topology': topo,
        'Sim Type': sim,
        'Avg Time (ms)': avg_mean,
        'Max Time (ms)': max_mean,
        'Difference (ms)': max_mean - avg_mean,
        'Difference (%)': diff_pct
    })

avg_max_df = pd.DataFrame(comparison_avg_max)

print("--- How much higher is Completion Time vs Average Time? ---\n")
for sim_type in avg_max_df['Sim Type'].unique():
    sim_data = avg_max_df[avg_max_df['Sim Type'] == sim_type]
    print(f"{sim_type}:")
    print(f"  Average difference: {sim_data['Difference (%)'].mean():.2f}%")
    print(f"  Median difference: {sim_data['Difference (%)'].median():.2f}%")
    print(f"  Max difference: {sim_data['Difference (%)'].max():.2f}%")
    print()

# Visualization
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Average vs Completion Time', 'Difference Distribution')
)

for sim_type in avg_max_df['Sim Type'].unique():
    sim_data = avg_max_df[avg_max_df['Sim Type'] == sim_type]
    color = 'blue' if sim_type == 'G2' else 'red'
    
    fig.add_trace(go.Scatter(
        x=sim_data['Avg Time (ms)'],
        y=sim_data['Max Time (ms)'],
        mode='markers',
        name=sim_type,
        marker=dict(color=color, size=8, opacity=0.6)
    ), row=1, col=1)

# Add diagonal line
min_val = avg_max_df[['Avg Time (ms)', 'Max Time (ms)']].min().min()
max_val = avg_max_df[['Avg Time (ms)', 'Max Time (ms)']].max().max()
fig.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    name='y=x',
    line=dict(dash='dash', color='gray'),
    showlegend=False
), row=1, col=1)

# Histogram of differences
for sim_type in avg_max_df['Sim Type'].unique():
    sim_data = avg_max_df[avg_max_df['Sim Type'] == sim_type]
    color = 'blue' if sim_type == 'G2' else 'red'
    
    fig.add_trace(go.Histogram(
        x=sim_data['Difference (%)'],
        name=f'{sim_type} Diff',
        marker_color=color,
        opacity=0.6,
        showlegend=False
    ), row=1, col=2)

fig.update_xaxes(title_text="Average Time (ms)", row=1, col=1)
fig.update_yaxes(title_text="Completion Time (ms)", row=1, col=1)
fig.update_xaxes(title_text="Difference (%)", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=2)

fig.update_layout(
    title='Comparison: Average Time vs Completion Time (Max NPU)',
    height=500,
    showlegend=True
)

fig.show()

print(f"\n{'='*80}\n")



DETAILED STATISTICAL ANALYSIS: Average vs Completion Time


METRIC: Average Time

--- Summary Statistics for Average Time ---

G2 Variability:
  Average CV: 13.019%
  Std Dev of CV: 0.697%
  Max CV: 13.512%

NS3 Variability:
  Average CV: 11.293%
  Std Dev of CV: 1.400%
  Max CV: 12.283%

Divergence Analysis:
  Mean Divergence: 3.39%
  Best Case (Best NS3 vs G2): -13.73%
  Worst Case (Worst NS3 vs G2): 26.77%


--- Top 10 Configurations by NS3 Variability (CV) for Average Time ---


,Workload,Group,Topology,G2 Mean (ms),G2 Std (ms),G2 CV (%),G2 Runs,NS3 Mean (ms),NS3 Std (ms),NS3 CV (%),NS3 Runs,Mean Divergence (%),Best NS3 vs G2 Mean (%),Worst NS3 vs G2 Mean (%)
0,multiple_collectives_tp/1_collectives_all_gather,-1,Dragonfly_16_ECMP_ECMP,190.840981,23.905484,12.526389,20,210.161996,25.814088,12.282948,20,10.124144,-9.273854,36.757777
1,multiple_collectives_tp/3_collectives_all_gath...,-1,Dragonfly_16_ECMP_ECMP,751.236264,101.508777,13.512231,18,726.187634,74.817759,10.302814,18,-3.334321,-18.181479,16.779496



METRIC: Completion Time

--- Summary Statistics for Completion Time ---

G2 Variability:
  Average CV: 16.802%
  Std Dev of CV: 4.418%
  Max CV: 19.926%

NS3 Variability:
  Average CV: 14.006%
  Std Dev of CV: 4.898%
  Max CV: 17.469%

Divergence Analysis:
  Mean Divergence: 3.79%
  Best Case (Best NS3 vs G2): -18.31%
  Worst Case (Worst NS3 vs G2): 36.98%


--- Top 10 Configurations by NS3 Variability (CV) for Completion Time ---


,Workload,Group,Topology,G2 Mean (ms),G2 Std (ms),G2 CV (%),G2 Runs,NS3 Mean (ms),NS3 Std (ms),NS3 CV (%),NS3 Runs,Mean Divergence (%),Best NS3 vs G2 Mean (%),Worst NS3 vs G2 Mean (%)
0,multiple_collectives_tp/1_collectives_all_gather,-1,Dragonfly_16_ECMP_ECMP,210.959292,42.036131,19.926181,20,235.892551,41.208688,17.469262,20,11.818991,-17.926009,57.068738
1,multiple_collectives_tp/3_collectives_all_gath...,-1,Dragonfly_16_ECMP_ECMP,782.800626,107.075145,13.678470,18,749.608642,79.023647,10.541987,18,-4.240158,-18.693150,16.900514



COMPARISON: Average Time vs Completion Time (Max)

--- How much higher is Completion Time vs Average Time? ---

ANALYTICAL_UNAWARE:
  Average difference: 0.00%
  Median difference: 0.00%
  Max difference: 0.00%

G2:
  Average difference: 5.99%
  Median difference: 4.20%
  Max difference: 10.54%

NS3:
  Average difference: 7.73%
  Median difference: 7.73%
  Max difference: 12.24%



In [24]:
# --- PER-NPU DETAILED ANALYSIS ---

print(f"\n{'='*80}")
print("PER-NPU COMPLETION TIME ANALYSIS")
print(f"{'='*80}\n")

# Analyze individual NPU completion times for selected configurations
unique_configs = comp_df.groupby(['workload', 'group_id', 'topology']).size().reset_index()[['workload', 'group_id', 'topology']]

print(f"Analyzing per-NPU completion times for up to 5 configurations...\n")
print("Note: Each NPU's completion time is the LAST (max) time across all its communications\n")

for idx, config in unique_configs.head(5).iterrows():
    wl, gid, topo = config['workload'], config['group_id'], config['topology']
    
    config_df = comp_df[
        (comp_df['workload'] == wl) &
        (comp_df['group_id'] == gid) &
        (comp_df['topology'] == topo)
    ]
    
    g2_runs = config_df[config_df['sim_type'] == 'G2'].sort_values('run_number')
    ns3_runs = config_df[config_df['sim_type'] == 'NS3'].sort_values('run_number')
    
    if g2_runs.empty or ns3_runs.empty:
        continue
    
    print(f"\n{'='*60}")
    print(f"Configuration: {wl} | Group: {gid} | Topology: {topo}")
    print(f"{'='*60}\n")
    
    # Collect NPU completion times (last time for each NPU) across all runs
    g2_npu_data = []
    ns3_npu_data = []
    
    for _, run in g2_runs.iterrows():
        timing_file = find_timing_file(run['path'], 'G2')
        if timing_file:
            try:
                df_timing = pd.read_csv(timing_file)
                df_timing = df_timing[df_timing['callback_tick'] > 100].copy()
                # Get last (max) time for each NPU
                npu_completion = df_timing.groupby('sys_id')['callback_tick'].max().reset_index()
                npu_completion['run_number'] = run['run_number']
                npu_completion['time_ms'] = npu_completion['callback_tick'] / 1_000_000
                g2_npu_data.append(npu_completion[['sys_id', 'run_number', 'time_ms', 'callback_tick']])
            except Exception as e:
                print(f"Error reading G2 timing file: {e}")
    
    for _, run in ns3_runs.iterrows():
        timing_file = find_timing_file(run['path'], 'NS3')
        if timing_file:
            try:
                df_timing = pd.read_csv(timing_file)
                if 'node_name' in df_timing.columns:
                    df_timing = df_timing[df_timing['node_name'] != 'dummy_node']
                df_timing = df_timing[df_timing['callback_tick'] > 100].copy()
                # Get last (max) time for each NPU
                npu_completion = df_timing.groupby('sys_id')['callback_tick'].max().reset_index()
                npu_completion['run_number'] = run['run_number']
                npu_completion['time_ms'] = npu_completion['callback_tick'] / 1_000_000
                npu_completion['ecmp_seed'] = run['ecmp_seed']
                ns3_npu_data.append(npu_completion[['sys_id', 'run_number', 'time_ms', 'callback_tick', 'ecmp_seed']])
            except Exception as e:
                print(f"Error reading NS3 timing file: {e}")
    
    if not g2_npu_data or not ns3_npu_data:
        continue
    
    g2_all = pd.concat(g2_npu_data, ignore_index=True)
    ns3_all = pd.concat(ns3_npu_data, ignore_index=True)
    
    # Statistics per NPU across all runs (these are now completion times)
    g2_per_npu = g2_all.groupby('sys_id')['time_ms'].agg(['mean', 'std', 'min', 'max', 'count']).reset_index()
    g2_per_npu['cv'] = (g2_per_npu['std'] / g2_per_npu['mean'] * 100)
    
    ns3_per_npu = ns3_all.groupby('sys_id')['time_ms'].agg(['mean', 'std', 'min', 'max', 'count']).reset_index()
    ns3_per_npu['cv'] = (ns3_per_npu['std'] / ns3_per_npu['mean'] * 100)
    
    print(f"G2 NPU Completion Statistics (across {len(g2_runs)} runs):")
    print(f"  Average CV per NPU: {g2_per_npu['cv'].mean():.3f}%")
    print(f"  Max CV: {g2_per_npu['cv'].max():.3f}% (NPU {g2_per_npu.loc[g2_per_npu['cv'].idxmax(), 'sys_id']})")
    print(f"  Range of mean completion times: {g2_per_npu['mean'].min():.2f} - {g2_per_npu['mean'].max():.2f} ms")
    print(f"  Load imbalance (range/mean): {((g2_per_npu['mean'].max() - g2_per_npu['mean'].min()) / g2_per_npu['mean'].mean() * 100):.2f}%")
    print()
    
    print(f"NS3 NPU Completion Statistics (across {len(ns3_runs)} runs):")
    print(f"  Average CV per NPU: {ns3_per_npu['cv'].mean():.3f}%")
    print(f"  Max CV: {ns3_per_npu['cv'].max():.3f}% (NPU {ns3_per_npu.loc[ns3_per_npu['cv'].idxmax(), 'sys_id']})")
    print(f"  Range of mean completion times: {ns3_per_npu['mean'].min():.2f} - {ns3_per_npu['mean'].max():.2f} ms")
    print(f"  Load imbalance (range/mean): {((ns3_per_npu['mean'].max() - ns3_per_npu['mean'].min()) / ns3_per_npu['mean'].mean() * 100):.2f}%")
    print()
    
    # Create visualization
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Per-NPU Mean Completion Time (All Runs)',
            'Per-NPU CV Comparison',
            'Per-NPU Completion Time Distribution',
            'Overall vs Average Completion Time per Run'
        ),
        specs=[[{"secondary_y": False}, {"secondary_y": False}],
               [{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    # 1. Per-NPU mean completion times
    fig.add_trace(go.Scatter(
        x=g2_per_npu['sys_id'],
        y=g2_per_npu['mean'],
        mode='markers',
        name='G2',
        marker=dict(size=10, color='blue'),
        error_y=dict(type='data', array=g2_per_npu['std'], visible=True)
    ), row=1, col=1)
    
    fig.add_trace(go.Scatter(
        x=ns3_per_npu['sys_id'],
        y=ns3_per_npu['mean'],
        mode='markers',
        name='NS3',
        marker=dict(size=10, color='red'),
        error_y=dict(type='data', array=ns3_per_npu['std'], visible=True)
    ), row=1, col=1)
    
    # 2. CV per NPU
    fig.add_trace(go.Bar(
        x=g2_per_npu['sys_id'],
        y=g2_per_npu['cv'],
        name='G2 CV',
        marker_color='lightblue',
        showlegend=False
    ), row=1, col=2)
    
    fig.add_trace(go.Bar(
        x=ns3_per_npu['sys_id'],
        y=ns3_per_npu['cv'],
        name='NS3 CV',
        marker_color='lightcoral',
        showlegend=False
    ), row=1, col=2)
    
    # 3. Box plot of all NPU completion times
    fig.add_trace(go.Box(
        y=g2_all['time_ms'],
        x=['G2'] * len(g2_all),
        name='G2',
        marker_color='blue',
        boxmean='sd'
    ), row=2, col=1)
    
    fig.add_trace(go.Box(
        y=ns3_all['time_ms'],
        x=['NS3'] * len(ns3_all),
        name='NS3',
        marker_color='red',
        boxmean='sd'
    ), row=2, col=1)
    
    # 4. Overall completion (max) vs average completion per run
    g2_run_stats = g2_runs[['run_number', 'avg_time', 'max_time']].copy()
    g2_run_stats['avg_ms'] = g2_run_stats['avg_time'] / 1_000_000
    g2_run_stats['max_ms'] = g2_run_stats['max_time'] / 1_000_000
    
    ns3_run_stats = ns3_runs[['run_number', 'avg_time', 'max_time']].copy()
    ns3_run_stats['avg_ms'] = ns3_run_stats['avg_time'] / 1_000_000
    ns3_run_stats['max_ms'] = ns3_run_stats['max_time'] / 1_000_000
    
    fig.add_trace(go.Scatter(
        x=g2_run_stats['avg_ms'],
        y=g2_run_stats['max_ms'],
        mode='markers',
        name='G2',
        marker=dict(size=12, color='blue'),
        text=[f"Run {r}" for r in g2_run_stats['run_number']],
        showlegend=False
    ), row=2, col=2)
    
    fig.add_trace(go.Scatter(
        x=ns3_run_stats['avg_ms'],
        y=ns3_run_stats['max_ms'],
        mode='markers',
        name='NS3',
        marker=dict(size=12, color='red'),
        text=[f"Run {r}" for r in ns3_run_stats['run_number']],
        showlegend=False
    ), row=2, col=2)
    
    # Diagonal line
    all_times = pd.concat([g2_run_stats[['avg_ms', 'max_ms']], ns3_run_stats[['avg_ms', 'max_ms']]])
    min_val = all_times.min().min()
    max_val = all_times.max().max()
    fig.add_trace(go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode='lines',
        line=dict(dash='dash', color='gray'),
        showlegend=False
    ), row=2, col=2)
    
    # Update axes
    fig.update_xaxes(title_text="NPU ID", row=1, col=1)
    fig.update_yaxes(title_text="Mean Completion Time (ms)", row=1, col=1)
    fig.update_xaxes(title_text="NPU ID", row=1, col=2)
    fig.update_yaxes(title_text="CV (%)", row=1, col=2)
    fig.update_yaxes(title_text="Completion Time (ms)", row=2, col=1)
    fig.update_xaxes(title_text="Average Completion Time (ms)", row=2, col=2)
    fig.update_yaxes(title_text="Overall Completion Time (ms)", row=2, col=2)
    
    fig.update_layout(
        title=f"Per-NPU Completion Time Analysis: {wl} | Group {gid} | {topo}",
        height=800,
        showlegend=True
    )
    
    fig.show()



PER-NPU COMPLETION TIME ANALYSIS

Analyzing per-NPU completion times for up to 5 configurations...

Note: Each NPU's completion time is the LAST (max) time across all its communications


Configuration: multiple_collectives_tp/1_collectives_all_gather | Group: -1 | Topology: Dragonfly_16_ECMP_ECMP

G2 NPU Completion Statistics (across 20 runs):
  Average CV per NPU: 14.818%
  Max CV: 19.247% (NPU 1)
  Range of mean completion times: 186.07 - 196.07 ms
  Load imbalance (range/mean): 5.24%

NS3 NPU Completion Statistics (across 20 runs):
  Average CV per NPU: 14.054%
  Max CV: 19.687% (NPU 6)
  Range of mean completion times: 191.18 - 223.29 ms
  Load imbalance (range/mean): 15.28%




Configuration: multiple_collectives_tp/3_collectives_all_gather_all_gather_reduce_scatter | Group: -1 | Topology: Dragonfly_16_ECMP_ECMP

G2 NPU Completion Statistics (across 18 runs):
  Average CV per NPU: 13.779%
  Max CV: 14.375% (NPU 4)
  Range of mean completion times: 738.57 - 762.86 ms
  Load imbalance (range/mean): 3.23%

NS3 NPU Completion Statistics (across 18 runs):
  Average CV per NPU: 10.504%
  Max CV: 10.997% (NPU 11)
  Range of mean completion times: 713.71 - 739.54 ms
  Load imbalance (range/mean): 3.56%

